In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

api_key = dbutils.secrets.get(scope="iot-scope", key="api-key")
raw_path = "/FileStore/tables/iot_suvm/raw/"

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *

df_sensor = spark.read.json("/FileStore/tables/iot_suvm/raw/sensor_readings.json", multiLine=True).select("*", "_metadata.file_path") 

df_bronze = df_sensor.withColumn("_ingested_at", current_timestamp()).withColumnRenamed("file_path", "_source_file")

df_bronze.write.format("delta").mode("overwrite").saveAsTable("iot_suvm_catalog.bronze.sensor_readings")

spark.read.json("/FileStore/tables/iot_suvm/raw/devices.json", multiLine=True).write.format("delta").mode("overwrite").saveAsTable("iot_suvm_catalog.bronze.devices")

spark.read.json("/FileStore/tables/iot_suvm/raw/locations.json", multiLine=True).write.format("delta").mode("overwrite").saveAsTable("iot_suvm_catalog.bronze.locations")

spark.sql("ALTER TABLE iot_suvm_catalog.bronze.sensor_readings ADD CONSTRAINT temp_range CHECK (temperature_c BETWEEN -50 AND 200)")


In [0]:
data = [("sensor-001", "HOT_DATA", "2024-01-01")] 
bad_df = spark.createDataFrame(data, ["device_id", "temperature_c", "reading_ts"])
bad_df.write.format("delta").mode("append").saveAsTable("iot_suvm_catalog.bronze.sensor_readings")

In [0]:
spark.table("iot_suvm_catalog.bronze.sensor_readings").createOrReplaceTempView("v_bronze_sensors")
spark.table("iot_suvm_catalog.bronze.devices").createOrReplaceTempView("v_devices")
spark.table("iot_suvm_catalog.bronze.locations").createOrReplaceTempView("v_locations")

final_silver_sql = """
WITH transformed_data AS (
    SELECT 
        *,
        CAST(reading_ts AS TIMESTAMP) AS reading_ts_cast,
        CAST(reading_ts AS DATE) AS reading_date,
        (temperature_c > 90 OR vibration_hz > 3.0) AS is_anomaly
    FROM v_bronze_sensors
),
windowed_data AS (
    SELECT 
        *,
        AVG(temperature_c) OVER (
            PARTITION BY device_id 
            ORDER BY reading_ts_cast 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_avg_temp
    FROM transformed_data
)
SELECT 
    w.*, 
    d.device_type, 
    l.zone_name, 
    l.floor, 
    l.supervisor
FROM windowed_data w
JOIN v_devices d ON w.device_id = d.device_id
JOIN v_locations l ON w.location_id = l.location_id
"""

final_silver = spark.sql(final_silver_sql)

final_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("location_id").saveAsTable("iot_suvm_catalog.silver.sensor_readings")


spark.sql("OPTIMIZE iot_suvm_catalog.silver.sensor_readings ZORDER BY (device_id, reading_ts_cast)")

In [0]:
%sql
USE CATALOG iot_suvm_catalog;

CREATE OR REPLACE VIEW gold.hourly_device_health AS
SELECT 
    device_id,
    date_trunc('hour', reading_ts) AS reading_hour,
    AVG(temperature_c) AS avg_temp,
    AVG(vibration_hz) AS avg_vibration,
    COUNT(CASE WHEN is_anomaly = true THEN 1 END) AS anomaly_count
FROM silver.sensor_readings
GROUP BY device_id, reading_hour;

CREATE OR REPLACE VIEW gold.critical_alerts AS
SELECT 
    reading_id, device_id, reading_ts, temperature_c, status, zone_name, supervisor
FROM silver.sensor_readings 
WHERE status = 'critical'
ORDER BY reading_ts DESC;

SELECT * FROM gold.hourly_device_health LIMIT 10;

In [0]:
streaming_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .load(raw_path))

prepared_df = (streaming_df
    .selectExpr("*", "CAST(reading_ts AS TIMESTAMP) AS ts")
    .selectExpr("*", "window(ts, '1 minute') AS time_window")
    .withWatermark("ts", "5 minutes"))

agg_stream = (prepared_df
    .groupBy("time_window", "device_id")
    .agg({"temperature_c": "max"})
    .withColumnRenamed("max(temperature_c)", "max_temp")
)

query = (agg_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .toTable("iot_suvm_catalog.gold.streaming_device_summary"))

In [0]:
def workflow_summary():
    return {
        "Job": "IoT_Sensor_Pipeline",
        "Catalog": "iot_suvm_catalog",
        "Schedule": "Hourly",
        "Tasks": ["Bronze_Ingest", "Silver_Transform", "Gold_Views", "Stream_Agg"]
    }
print(workflow_summary())

In [0]:
%sql
USE CATALOG iot_suvm_catalog;

CREATE OR REPLACE FUNCTION bronze.supervisor_mask(supervisor STRING)
RETURN CASE 
  WHEN is_account_group_member('supervisor_access') THEN supervisor 
  ELSE '#### CONFIDENTIAL ####' 
END;

ALTER TABLE bronze.locations ALTER COLUMN supervisor SET MASK bronze.supervisor_mask;

CREATE OR REPLACE FUNCTION silver.floor_filter(floor_num INT)
RETURN floor_num = 1; 

ALTER TABLE silver.sensor_readings SET ROW FILTER silver.floor_filter ON (floor);

SELECT "Governance policies applied successfully" AS status;

In [0]:
%sql
-- Check Column Masking
SELECT supervisor FROM iot_suvm_catalog.bronze.locations LIMIT 5;
-- (Agar aap admin hain aur group mein nahi hain, toh aapko '#### CONFIDENTIAL ####' dikhna chahiye)

-- Check Row Filtering
SELECT DISTINCT floor FROM iot_suvm_catalog.silver.sensor_readings;
-- (Yahan sirf floor '1' dikhna chahiye agar filter sahi se apply hua hai)